# Expandiffusion on Google Colab

Use **Runtime > Run all**. Colab will pause once to ask for an optional Hugging Face token. Paste it if you need gated models such as FLUX, or press Enter to skip.

After that, the notebook clones the repo, installs dependencies, starts the API and web UI, prints the temporary Colab URLs, and leaves the app running.

In [ ]:
import os
from getpass import getpass

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token (optional, press Enter to skip): ").strip()

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

print("HF_TOKEN set:", bool(HF_TOKEN))

In [ ]:
import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from IPython.display import HTML, display
from google.colab import output

REPO_URL = "https://github.com/PailletJuanPablo/expandiffusion.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/expandiffusion")
API_PORT = 8011
WEB_PORT = 5180


def run(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)


def wait_for_json(url, label, timeout_seconds=180):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                return json.loads(response.read().decode("utf-8"))
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{label} did not become ready at {url}: {last_error}")


def wait_for_text(url, label, timeout_seconds=180):
    deadline = time.time() + timeout_seconds
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                return response.status, response.read(256).decode("utf-8", errors="replace")
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"{label} did not become ready at {url}: {last_error}")


print("Checking runtime hardware...")
subprocess.run(["nvidia-smi"], check=False)

if (PROJECT_DIR / "apps" / "api").exists():
    print(f"Using existing checkout at {PROJECT_DIR}")
else:
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(PROJECT_DIR)])

api_dir = PROJECT_DIR / "apps" / "api"
web_dir = PROJECT_DIR / "apps" / "web"

run([sys.executable, "-m", "pip", "install", "-e", ".[dev,diffusers]"], cwd=api_dir)
run(["npm", "--prefix", str(web_dir), "ci"])

env_values = {
    "EXPANDIFFUSION_PLUGIN_DIR": "apps/api/plugins",
    "EXPANDIFFUSION_API_HOST": "0.0.0.0",
    "EXPANDIFFUSION_API_PORT": str(API_PORT),
    "EXPANDIFFUSION_WEB_HOST": "0.0.0.0",
    "EXPANDIFFUSION_WEB_PORT": str(WEB_PORT),
    "EXPANDIFFUSION_WEB_ALLOWED_HOSTS": "*",
    "EXPANDIFFUSION_PYTHON": sys.executable,
}
for key, value in env_values.items():
    os.environ[key] = value

(PROJECT_DIR / ".env").write_text(
    "\n".join(f"{key}={value}" for key, value in env_values.items()) + "\n",
    encoding="utf-8",
)

if "EXPANDIFFUSION_PROCESS" in globals() and EXPANDIFFUSION_PROCESS.poll() is None:
    EXPANDIFFUSION_PROCESS.terminate()
    EXPANDIFFUSION_PROCESS.wait(timeout=20)

log_dir = PROJECT_DIR / "colab-logs"
log_dir.mkdir(exist_ok=True)
log_file = open(log_dir / "dev.log", "w", encoding="utf-8")
EXPANDIFFUSION_PROCESS = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=PROJECT_DIR,
    env=os.environ.copy(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)

api_health = wait_for_json(f"http://127.0.0.1:{API_PORT}/api/health", "Expandiffusion API")
if not api_health.get("ok"):
    raise RuntimeError(f"API health check failed: {api_health}")

web_status, _web_preview = wait_for_text(f"http://127.0.0.1:{WEB_PORT}/", "Expandiffusion web UI")
api_url = output.eval_js(f"google.colab.kernel.proxyPort({API_PORT})")
ui_url = output.eval_js(f"google.colab.kernel.proxyPort({WEB_PORT})")
if not isinstance(api_url, str) or not api_url.startswith("http"):
    raise RuntimeError(f"Colab did not return a valid API proxy URL: {api_url!r}")
if not isinstance(ui_url, str) or not ui_url.startswith("http"):
    raise RuntimeError(f"Colab did not return a valid UI proxy URL: {ui_url!r}")

print("API ready:", bool(api_health.get("ok")))
print("Local web status:", web_status)
print("Temporary Colab API URL:", api_url)
print("Temporary Colab UI URL:", ui_url)
display(HTML(f'<p><a href="{ui_url}" target="_blank" rel="noopener noreferrer">Open Expandiffusion UI</a></p>'))